In [1]:
import pandas as pd
from pathlib import Path

# File paths
PRODUCTS_PATH = "../amazon_dataset/products/amazon_products.csv"
CATEGORIES_PATH = "../amazon_dataset/products/amazon_categories.csv"
REVIEWS_FILES = [
    "../amazon_dataset/reviews/Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv",
    "../amazon_dataset/reviews/Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv",
    "../amazon_dataset/reviews/1429_1.csv",
]

# Load and clean product dataset
df_products = pd.read_csv(PRODUCTS_PATH)
df_products = df_products.dropna(subset=["asin", "title", "price", "listPrice"])
df_products["asin"] = df_products["asin"].astype(str).str.strip()
df_products["no_of_reviews"] = df_products["reviews"]
df_products = df_products.drop(columns=["reviews"])

# Load and merge categories
df_categories = pd.read_csv(CATEGORIES_PATH)
df_categories = df_categories.rename(columns={"id": "category_id", "category_name": "category"})
df_categories["category_id"] = df_categories["category_id"].astype(int)
df_products["category_id"] = df_products["category_id"].astype(int)
df_products = df_products.merge(df_categories, on="category_id", how="left")

# Combine all review files
review_dfs = []
for path in REVIEWS_FILES:
    df = pd.read_csv(path)
    df = df.dropna(subset=["asins", "reviews.text"])
    df["asins"] = df["asins"].astype(str).str.strip().str.split(",")
    df = df.explode("asins")
    df["asins"] = df["asins"].str.strip()
    review_dfs.append(df)

# Concatenate all review dataframes
df_reviews = pd.concat(review_dfs, ignore_index=True)
print(f"✅ Combined reviews dataset size: {len(df_reviews):,} rows")

# Optional: save if needed
# df_reviews.to_csv("all_reviews_combined.csv", index=False)


/var/folders/lg/kf5t1j251mlfvtmggk4d1qh40000gn/T/ipykernel_99014/1840868789.py:30: DtypeWarning: Columns (1,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


✅ Combined reviews dataset size: 93,050 rows


In [2]:
df = df_reviews.dropna(subset=["asins", "reviews.text", "reviews.title"])

# Combine review title and text into a single string
df["review_combined"] = df["reviews.title"].astype(str) + " - " + df["reviews.text"].astype(str)

# Group by ASIN and aggregate reviews into a list
new_df_reviews = df.groupby("asins")["review_combined"].apply(list).reset_index()

# Result: Each row has an 'asins' and a list of review strings
print(new_df_reviews.head())

        asins                                    review_combined
0  B0002LCUZK  [Bottom expands as well - This folder also exp...
1  B001NIZB5M  [Kindle Power Adapter Specs - Since the detail...
2  B002Y27P3M  [Worth the money. Not perfect, but very very g...
3  B002Y27P6Y  [Heavy but more protective than a slipcase One...
4  B005OOKNP4  [An excellent keyboard -- with one significant...


/var/folders/lg/kf5t1j251mlfvtmggk4d1qh40000gn/T/ipykernel_99014/55073627.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["review_combined"] = df["reviews.title"].astype(str) + " - " + df["reviews.text"].astype(str)


In [3]:
# Combine title and text (handling missing titles)
df_reviews = df_reviews.dropna(subset=["reviews.title"])
df_reviews["review_combined"] = (
    df_reviews["reviews.title"].astype(str) + " - " + df_reviews["reviews.text"].astype(str)
)

# Group by ASIN
new_df_reviews = df_reviews.groupby("asins")["review_combined"].apply(list).reset_index()
new_df_reviews = new_df_reviews.rename(columns={"asins": "asin"})
new_df_reviews["asin"] = new_df_reviews["asin"].astype(str).str.strip()

# Merge into products
df_products["asin"] = df_products["asin"].astype(str).str.strip()
df_merged = pd.merge(df_products, new_df_reviews, on="asin", how="left")

print(f"✅ Merged dataset size: {len(df_merged)} products")
print(f"📝 Products with reviews: {df_merged['review_combined'].notnull().sum()}")


✅ Merged dataset size: 1426336 products
📝 Products with reviews: 9


In [4]:
df_merged["review_combined"].notnull().sum()

np.int64(9)

In [5]:
import os
import json
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

# Paths
QA_FOLDER = "../amazon_dataset/qa_dataset"
qa_by_asin = defaultdict(list)

import ast

def collect_qa_from_file(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    item = ast.literal_eval(line.strip())
                    asin = item.get("asin", "").strip()
                    if not asin or "questions" not in item:
                        continue
                    for q in item["questions"]:
                        q_text = q.get("questionText", "").strip()
                        for ans in q.get("answers", []):
                            a_text = ans.get("answerText", "").strip()
                            if q_text and a_text:
                                qa_by_asin[asin].append({
                                    "question": q_text,
                                    "answer": a_text
                                })
                except Exception as line_err:
                    print(f"⚠️ Line parse error in {file_path}: {line_err}")
    except Exception as e:
        print(f"⚠️ Failed to open {file_path}: {e}")


# Recursively walk through the QA folder
for root, _, files in os.walk(QA_FOLDER):
    for file in files:
        if file.endswith(".json"):
            full_path = os.path.join(root, file)
            collect_qa_from_file(full_path)

print(f"✅ Collected QA for {len(qa_by_asin)} ASINs")


✅ Collected QA for 171800 ASINs


In [6]:
df_qas = pd.DataFrame([
    {"asin": asin, "qa": json.dumps(qa_list)}  # store list of QAs as JSON string
    for asin, qa_list in qa_by_asin.items()
])

# Step 3: Clean and normalize ASINs
df_qas["asin"] = df_qas["asin"].astype(str).str.strip()
df_merged["asin"] = df_merged["asin"].astype(str).str.strip()

# Step 4: Merge with the main dataset
df_merged_with_qa = pd.merge(df_merged, df_qas, on="asin", how="left")

# ✅ Result: `df_merged_with_qa` now has an extra column `qa` with list of Q&A JSON strings
print(f"✅ Final dataset: {len(df_merged_with_qa)} rows")
print(f"🧠 Products with Q&A: {df_merged_with_qa['qa'].notnull().sum()}")

✅ Final dataset: 1426336 rows
🧠 Products with Q&A: 10441


In [7]:
df_merged_with_qa[df_merged_with_qa["qa"].notnull()].head()["qa"]



1772    [{"question": "How many pockets does this jack...
1971    [{"question": "can I play cricket with this?",...
2946    [{"question": "i wear a womens size 11. what s...
3264    [{"question": "I'm a 31 waist. Should I buy sm...
3996    [{"question": "size my son is 14 years old, he...
Name: qa, dtype: object

In [8]:
import json

def group_qas_in_batches(qa_str, batch_size=5):
    try:
        qa_list = json.loads(qa_str)
        grouped = []

        for i in range(0, len(qa_list), batch_size):
            batch = qa_list[i:i + batch_size]
            formatted = ""
            for j, qa in enumerate(batch, 1):
                formatted += f"QA {j}:\nQ: {qa['question']}\nA: {qa['answer']}\n\n"
            grouped.append(formatted.strip())
        return grouped
    except (json.JSONDecodeError, TypeError):
        return None  # If string is invalid or NaN

# Apply to your merged dataframe
df_merged_with_qa["qa"] = df_merged_with_qa["qa"].apply(group_qas_in_batches)


In [9]:
df_merged_with_qa[df_merged_with_qa["qa"].notnull()].iloc[0]["qa"]


['QA 1:\nQ: How many pockets does this jacket have?\nA: Side pocket on each side that empties into mesh enclosure.  Total 2.\n\nQA 2:\nQ: How many pockets does this jacket have?\nA: Two zipped outer pockets.\n\nQA 3:\nQ: How many pockets does this jacket have?\nA: Two pockets that zip.\n\nQA 4:\nQ: How many pockets does this jacket have?\nA: Four pockets i wish it had one on the right side of the jacket\n\nQA 5:\nQ: Is this jacket warm enough for New England winter?\nA: This is a good warm weather jacket for biking or hiking. In cold New Orleans weather it is a good wind breaker. Its main function for us is as a raincoat. It is a good raincoat or a good second layer for bikiing when the temp is in the 50s. It does not provide any real winter warmth. I have lived in Maine. This would be a summer coat.',
 "QA 1:\nQ: Is this jacket warm enough for New England winter?\nA: This is a raincoat only. A shell. IT works very well in the rain but wouldn't keep you warm beyond a 50 degree day. My 

In [10]:
import json
import re

def chunk_qas(qa_data, chunk_size=5):
    """
    Convert QA data to chunks of structured Q&A pairs
    """
    try:
        # Handle None case
        if qa_data is None:
            return None
        
        # If it's already a list (which it is in your data)
        if isinstance(qa_data, list):
            qa_list = qa_data
        # If it's a JSON string, parse it
        elif isinstance(qa_data, str):
            try:
                qa_list = json.loads(qa_data)
            except json.JSONDecodeError:
                # If it's not valid JSON, treat as single string
                qa_list = [qa_data]
        else:
            return None
        
        # Parse each QA string to extract individual Q&A pairs
        all_qa_pairs = []
        
        for qa_text in qa_list:
            if not isinstance(qa_text, str) or not qa_text.strip():
                continue
            
            # Parse individual Q&A pairs from the text
            qa_pairs = parse_qa_text(qa_text)
            all_qa_pairs.extend(qa_pairs)
        
        # If no valid Q&A pairs found, return None
        if not all_qa_pairs:
            return None
        
        # Split into chunks of size `chunk_size`
        chunks = [all_qa_pairs[i:i + chunk_size] for i in range(0, len(all_qa_pairs), chunk_size)]
        return chunks
        
    except Exception as e:
        print(f"Error processing QA data: {e}")
        return None

def parse_qa_text(qa_text):
    """
    Parse Q&A text to extract individual question-answer pairs
    """
    qa_pairs = []
    
    # Split by "QA X:" pattern to get individual Q&A blocks
    qa_blocks = re.split(r'\n\nQA \d+:', qa_text)
    
    # Also handle the first block (which might not have the split pattern)
    if qa_blocks and qa_blocks[0].strip():
        # Check if first block starts with "QA X:"
        first_block = qa_blocks[0]
        if re.match(r'^QA \d+:', first_block):
            # Remove the "QA X:" prefix
            first_block = re.sub(r'^QA \d+:\s*', '', first_block)
            qa_blocks[0] = first_block
    
    for block in qa_blocks:
        if not block.strip():
            continue
            
        # Clean up the block
        block = block.strip()
        
        # Look for Q: and A: patterns
        q_match = re.search(r'Q:\s*(.+?)(?=\nA:)', block, re.DOTALL)
        a_match = re.search(r'A:\s*(.+?)(?=\n\nQA|\n\n\n|$)', block, re.DOTALL)
        
        if q_match and a_match:
            question = q_match.group(1).strip()
            answer = a_match.group(1).strip()
            
            # Clean up extra whitespace and formatting
            question = re.sub(r'\s+', ' ', question)
            answer = re.sub(r'\s+', ' ', answer)
            
            # Remove any trailing text like "Read More", "Show Less", etc.
            answer = re.sub(r'\s*»\s*Read More.*?«\s*Show Less\s*', '', answer, flags=re.DOTALL)
            answer = re.sub(r'\s*»\s*Read More.*$', '', answer, flags=re.DOTALL)
            answer = re.sub(r'\s*«\s*Show Less.*$', '', answer, flags=re.DOTALL)
            
            qa_pairs.append({
                'question': question,
                'answer': answer
            })
    
    return qa_pairs

# Apply to the 'qa' column
df_merged_with_qa["qa_chunks"] = df_merged_with_qa["qa"].apply(lambda x: chunk_qas(x, chunk_size=5))

In [11]:
# Check non-null counts individually
has_title = df_merged_with_qa["title"].notnull()
has_reviews = df_merged_with_qa["review_combined"].notnull()
has_qa = df_merged_with_qa["qa"].notnull()

# Count how many rows have each
print("Rows with title:", has_title.sum())
print("Rows with reviews:", has_reviews.sum())
print("Rows with QA:", has_qa.sum())

# Count how many have all three
print("Rows with all three (title, review, QA):", (has_title & has_reviews & has_qa).sum())

# Rows with title and reviews only
print("Rows with title and reviews (no QA needed):", (has_title & has_reviews).sum())

# Rows with title and QA only
print("Rows with title and QA (no review needed):", (has_title & has_qa).sum())


Rows with title: 1426336
Rows with reviews: 9
Rows with QA: 10441
Rows with all three (title, review, QA): 3
Rows with title and reviews (no QA needed): 9
Rows with title and QA (no review needed): 10441


In [12]:
# 1. Filter rows with reviews or QA (we must keep these)
has_reviews_or_qa = df_merged_with_qa["review_combined"].notnull() | df_merged_with_qa["qa"].notnull()
must_include_df = df_merged_with_qa[has_reviews_or_qa]

# 2. Get the number of rows we still need to reach 50,000
remaining_count = 11000 - len(must_include_df)

# 3. Filter the rest: rows without reviews or QA
remaining_pool = df_merged_with_qa[~has_reviews_or_qa]

# 4. Randomly sample the remaining rows (deterministic with random_state)
extra_rows = remaining_pool.sample(n=remaining_count, random_state=42)

# 5. Concatenate both parts
final_df = pd.concat([must_include_df, extra_rows], ignore_index=True)

# 6. Shuffle the final DataFrame if needed
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

# # 7. Save to file
final_df.to_json("../processed/final_10k_dataset.jsonl", orient="records", lines=True)


final_df.head()

,asin,title,imgUrl,productURL,stars,price,listPrice,category_id,isBestSeller,boughtInLastMonth,no_of_reviews,category,review_combined,qa,qa_chunks
0,B000PD0HQE,Microsoft Xbox 360 Elite 120GB Console Bundle,https://m.media-amazon.com/images/I/41x3Emcwhz...,https://www.amazon.com/dp/B000PD0HQE,3.7,0.00,0.00,245,False,0,0,"Xbox 360 Games, Consoles & Accessories",NaN,[QA 1:\nQ: Does it come with all the games\nA:...,[[{'question': 'Does it come with all the game...
1,B001C44824,Sabre Red 52CFT30 Crossfire Stream (MK-4) Pepp...,https://m.media-amazon.com/images/I/51FC2aEYXj...,https://www.amazon.com/dp/B001C44824,4.8,19.99,21.46,205,False,100,0,Safety & Security,NaN,[QA 1:\nQ: case recommendations please for dut...,[[{'question': 'case recommendations please fo...
2,B007ZP1JGM,Complete Protein & Vitamin Shake Mix by Nature...,https://m.media-amazon.com/images/I/61TTnWFBoC...,https://www.amazon.com/dp/B007ZP1JGM,4.4,23.14,0.00,136,False,100,0,Sports Nutrition Products,NaN,[QA 1:\nQ: How many servings per 16 oz contain...,[[{'question': 'How many servings per 16 oz co...
3,B000WGBNCQ,CCL Security Products Brass Sesamee K437 4 Dia...,https://m.media-amazon.com/images/I/71oO7xr6-8...,https://www.amazon.com/dp/B000WGBNCQ,4.4,0.00,0.00,144,False,0,0,Material Handling Products,NaN,[QA 1:\nQ: will it fit gym lockers?\nA: I use ...,"[[{'question': 'will it fit gym lockers?', 'an..."
4,B0082B41FO,"Cosmetic Teeth - 1 Pack - Large, Bleached - Up...",https://m.media-amazon.com/images/I/51VFAp7eo1...,https://www.amazon.com/dp/B0082B41FO,3.3,40.00,0.00,126,False,0,0,Oral Care Products,NaN,[QA 1:\nQ: How do I know rather I should buy s...,[[{'question': 'How do I know rather I should ...


In [13]:
len(final_df)

NameError: name 'final_df' is not defined

In [ ]:
# from datasets import load_dataset

# # 1. Pick the config you need (e.g. “raw_meta_Electronics” or whatever applies)
# config_name = "raw_meta_Electronics"

# # 2. Stream the dataset, then immediately drop the unwanted list<struct> column
# ds = (
#     load_dataset(
#         "McAuley-Lab/Amazon-Reviews-2023",
#         config_name,
#         split="full",
#         streaming=True
#     )
#     .remove_columns(["images", "verified_purchase", "review_date", "review_title"])
#     # remove any other nested/list columns you won’t use
# )

# # 3. Now safely filter by your ASIN list
# target_asins = set(final_df["asin"])
# filtered = (
#     entry
#     for entry in ds
#     if entry["asin"] in target_asins
# )

# # 4. If you need to collect into a DataFrame, you can do:
# import pandas as pd

# # (but beware of loading too much into memory if the filtered set is large)
# filtered_list = list(filtered)
# df_filtered = pd.DataFrame(filtered_list)

# print(df_filtered.head())


KeyError: 'asin'

In [54]:
final_df.head(2)

,asin,title,imgUrl,productURL,stars,price,listPrice,category_id,isBestSeller,boughtInLastMonth,no_of_reviews,category,review_combined,qa,qa_chunks
0,B0B6LQRTGR,You Are My Sunshine - Sunflower Necklace Locke...,https://m.media-amazon.com/images/I/71WJX1s3Z2...,https://www.amazon.com/dp/B0B6LQRTGR,4.6,19.95,0.0,123,False,0,0,Women's Jewelry,NaN,NaN,None
1,B000W9TVEK,Convenience Concepts Designs2Go Classic Glass ...,https://m.media-amazon.com/images/I/81YUPIuSnl...,https://www.amazon.com/dp/B000W9TVEK,4.8,49.16,65.1,166,False,50,0,Furniture,NaN,"[{""question"": ""What is the max load capacity f...",[[{'question': 'What is the max load capacity ...


## Jsonl to csv

In [13]:
import json
import pandas as pd

input_file = "../../../processed/processed/documents_by_type/titles.jsonl"  # Your actual JSONL path
output_file = "../../../processed/processed/raw_data.csv"   # Your output CSV path

metadata_list = []

with open(input_file, 'r') as f:
    for line in f:
        if line.strip():  # skip empty lines
            record = json.loads(line)
            metadata = record.get("metadata", {})
            metadata_list.append(metadata)

# Create and save DataFrame
df = pd.DataFrame(metadata_list)
df.to_csv(output_file, index=False)

print(f"Saved CSV with {len(df)} rows to: {output_file}")


Saved CSV with 11000 rows to: ../../../processed/processed/raw_data.csv


In [14]:
record

{'id': '5c972f3e-ce8a-480c-87f0-5b842c865570',
 'metadata': {'asin': 'B0010L1C28',
  'content_type': 'title',
  'doc_id': 'title_B0010L1C28',
  'source': 'amazon_products',
  'imgUrl': 'https://m.media-amazon.com/images/I/412K3jOBqgS._AC_UL320_.jpg',
  'productURL': 'https://www.amazon.com/dp/B0010L1C28',
  'stars': 4.7,
  'price': 9.99,
  'listPrice': 0.0,
  'category_id': 170,
  'isBestSeller': False,
  'boughtInLastMonth': 1000,
  'no_of_reviews': 0,
  'category': 'Kitchen & Dining'},
 'page_content': 'Rubbermaid Ice Bin 12.1" x 5.5" x 6.12"',
 'type': 'Document'}

In [15]:
df

,asin,content_type,doc_id,source,imgUrl,productURL,stars,price,listPrice,category_id,isBestSeller,boughtInLastMonth,no_of_reviews,category
0,B000PD0HQE,title,title_B000PD0HQE,amazon_products,https://m.media-amazon.com/images/I/41x3Emcwhz...,https://www.amazon.com/dp/B000PD0HQE,3.7,0.00,0.00,245,False,0,0,"Xbox 360 Games, Consoles & Accessories"
1,B001C44824,title,title_B001C44824,amazon_products,https://m.media-amazon.com/images/I/51FC2aEYXj...,https://www.amazon.com/dp/B001C44824,4.8,19.99,21.46,205,False,100,0,Safety & Security
2,B007ZP1JGM,title,title_B007ZP1JGM,amazon_products,https://m.media-amazon.com/images/I/61TTnWFBoC...,https://www.amazon.com/dp/B007ZP1JGM,4.4,23.14,0.00,136,False,100,0,Sports Nutrition Products
3,B000WGBNCQ,title,title_B000WGBNCQ,amazon_products,https://m.media-amazon.com/images/I/71oO7xr6-8...,https://www.amazon.com/dp/B000WGBNCQ,4.4,0.00,0.00,144,False,0,0,Material Handling Products
4,B0082B41FO,title,title_B0082B41FO,amazon_products,https://m.media-amazon.com/images/I/51VFAp7eo1...,https://www.amazon.com/dp/B0082B41FO,3.3,40.00,0.00,126,False,0,0,Oral Care Products
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10995,B000I14F2K,title,title_B000I14F2K,amazon_products,https://m.media-amazon.com/images/I/61qOp1dkn0...,https://www.amazon.com/dp/B000I14F2K,4.6,0.00,0.00,214,False,0,0,Welding & Soldering
10996,B00GBCLG0E,title,title_B00GBCLG0E,amazon_products,https://m.media-amazon.com/images/I/91e1FkdPY+...,https://www.amazon.com/dp/B00GBCLG0E,4.1,999.99,0.00,243,False,0,0,"PlayStation 3 Games, Consoles & Accessories"
10997,B003WXGLS2,title,title_B003WXGLS2,amazon_products,https://m.media-amazon.com/images/I/51slmzwh5G...,https://www.amazon.com/dp/B003WXGLS2,4.6,3.81,0.00,205,True,0,0,Safety & Security
10998,B00076OC8S,title,title_B00076OC8S,amazon_products,https://m.media-amazon.com/images/I/61vV7cSNNC...,https://www.amazon.com/dp/B00076OC8S,4.7,30.99,0.00,228,False,0,1279,Sports & Outdoor Play Toys


In [16]:
import pandas as pd

df=pd.read_csv("../../../processed/processed/raw_data.csv")

df.head()


,asin,content_type,doc_id,source,imgUrl,productURL,stars,price,listPrice,category_id,isBestSeller,boughtInLastMonth,no_of_reviews,category
0,B000PD0HQE,title,title_B000PD0HQE,amazon_products,https://m.media-amazon.com/images/I/41x3Emcwhz...,https://www.amazon.com/dp/B000PD0HQE,3.7,0.00,0.00,245,False,0,0,"Xbox 360 Games, Consoles & Accessories"
1,B001C44824,title,title_B001C44824,amazon_products,https://m.media-amazon.com/images/I/51FC2aEYXj...,https://www.amazon.com/dp/B001C44824,4.8,19.99,21.46,205,False,100,0,Safety & Security
2,B007ZP1JGM,title,title_B007ZP1JGM,amazon_products,https://m.media-amazon.com/images/I/61TTnWFBoC...,https://www.amazon.com/dp/B007ZP1JGM,4.4,23.14,0.00,136,False,100,0,Sports Nutrition Products
3,B000WGBNCQ,title,title_B000WGBNCQ,amazon_products,https://m.media-amazon.com/images/I/71oO7xr6-8...,https://www.amazon.com/dp/B000WGBNCQ,4.4,0.00,0.00,144,False,0,0,Material Handling Products
4,B0082B41FO,title,title_B0082B41FO,amazon_products,https://m.media-amazon.com/images/I/51VFAp7eo1...,https://www.amazon.com/dp/B0082B41FO,3.3,40.00,0.00,126,False,0,0,Oral Care Products


In [6]:
df.head()

""
0
1
2
3
4
